In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [7]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define N (1 << 20) // 1,048,576 elementos

// 1. Definición del Kernel
__global__ void vectorAdd(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main(void) {
    size_t size = N * sizeof(float);

    // 2. Reserva de memoria en el Host (CPU)
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_c = (float*)malloc(size);

    // Inicialización de datos de entrada
    for (int i = 0; i < N; i++) {
        h_a[i] = sinf(i);
        h_b[i] = cosf(i);
    }

    // 3. Reserva de memoria en el Device (GPU)
    float *d_a, *d_b, *d_c;
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // 4. Copia de datos: Host -> Device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // 5. Configuración y lanzamiento del Kernel
    int blockSize = 256;
    int numBlocks = (N + blockSize - 1) / blockSize;
    vectorAdd<<<numBlocks, blockSize>>>(d_a, d_b, d_c, N);

    // 6. Copia de resultados: Device -> Host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // 7. Verificación de resultados en CPU
    printf("Verificación del primer elemento: c = %f (esperado: %f)\n",
           h_c[0], sinf(0) + cosf(0));
    printf("Verificación del último elemento: c[%d] = %f\n",
           N - 1, h_c[N - 1]);

    // 8. Liberación de memoria en ambos espacios
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    printf("Programa finalizado correctamente.\n");
    return 0;
}

Overwriting vector_add.cu


In [8]:
!nvcc -O3 vector\_add.cu -o vector\_add 
!./vector\_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Verificación del primer elemento: c = 1.000000 (esperado: 1.000000)
Verificación del último elemento: c[1048575] = 0.172421
Programa finalizado correctamente.
